In [ ]:
import tkinter as tk
from tkinter import filedialog, messagebox
import fitz  # PyMuPDF
import math

# --- 1. The Imposition Engine ---
def process_imposition(input_file, output_file, art_w, art_h, sheet_w, sheet_h, is_duplex, is_variable, layout_style):
    try:
        doc_in = fitz.open(input_file)
        doc_out = fitz.open()
        IN2PT = 72
        
        art_w_pt = float(art_w) * IN2PT
        art_h_pt = float(art_h) * IN2PT
        sheet_w_pt = float(sheet_w) * IN2PT
        sheet_h_pt = float(sheet_h) * IN2PT

        cols = int(sheet_w_pt // art_w_pt)
        rows = int(sheet_h_pt // art_h_pt)
        n_up = int(cols * rows)
        total_input_pages = int(doc_in.page_count)

        if n_up == 0:
            messagebox.showerror("Error", "Artwork is larger than the press sheet!")
            return

        margin_x = float((sheet_w_pt - (cols * art_w_pt)) / 2.0)
        margin_y = float((sheet_h_pt - (rows * art_h_pt)) / 2.0)

        print("-" * 30)
        print(f"Executing: {layout_style} ({n_up}-up)")
        side_mode = "Duplex" if is_duplex else "Simplex"
        print(f"Mode: {side_mode}")

        def draw_crop_marks(page):
            mark_len = 18
            offset = 9 
            color = (0, 0, 0) 
            width = 0.5       
            for c in range(cols + 1):
                x = margin_x + (c * art_w_pt)
                page.draw_line(fitz.Point(x, margin_y - offset), fitz.Point(x, margin_y - offset - mark_len), color=color, width=width)
                top_y = margin_y + (rows * art_h_pt)
                page.draw_line(fitz.Point(x, top_y + offset), fitz.Point(x, top_y + offset + mark_len), color=color, width=width)
            for r in range(rows + 1):
                y = margin_y + (r * art_h_pt)
                page.draw_line(fitz.Point(margin_x - offset, y), fitz.Point(margin_x - offset - mark_len, y), color=color, width=width)
                right_x = margin_x + (cols * art_w_pt)
                page.draw_line(fitz.Point(right_x + offset, y), fitz.Point(right_x + offset + mark_len, y), color=color, width=width)

        # --- LAYOUT LOGIC BRANCH ---
        if layout_style == "Step & Repeat":
            if is_duplex:
                # 1. Generate Front Sheet
                sheet_front = doc_out.new_page(width=sheet_w_pt, height=sheet_h_pt)
                for r in range(rows):
                    for c in range(cols):
                        front_x = margin_x + (c * art_w_pt)
                        y = margin_y + (r * art_h_pt)
                        r_front = fitz.Rect(front_x, y, front_x + art_w_pt, y + art_h_pt)
                        if total_input_pages > 0:
                            sheet_front.show_pdf_page(r_front, doc_in, 0)
                draw_crop_marks(sheet_front)
                
                # 2. Generate Back Sheet (Mirrored)
                sheet_back = doc_out.new_page(width=sheet_w_pt, height=sheet_h_pt)
                for r in range(rows):
                    for c in range(cols):
                        back_x = margin_x + ((cols - 1 - c) * art_w_pt) 
                        y = margin_y + (r * art_h_pt)
                        r_back = fitz.Rect(back_x, y, back_x + art_w_pt, y + art_h_pt)
                        if total_input_pages > 1:
                            sheet_back.show_pdf_page(r_back, doc_in, 1)
                draw_crop_marks(sheet_back)
            else:
                for page_idx in range(total_input_pages):
                    sheet = doc_out.new_page(width=sheet_w_pt, height=sheet_h_pt)
                    for r in range(rows):
                        for c in range(cols):
                            x0 = margin_x + (c * art_w_pt)
                            y0 = margin_y + (r * art_h_pt)
                            slot_rect = fitz.Rect(x0, y0, x0 + art_w_pt, y0 + art_h_pt)
                            sheet.show_pdf_page(slot_rect, doc_in, int(page_idx))
                    draw_crop_marks(sheet)
                            
        elif layout_style == "Cut & Stack":
            if is_duplex:
                total_physical_sheets = int(math.ceil(total_input_pages / float(n_up * 2)))
                pages_per_stack = int(total_physical_sheets * 2)
                
                for sheet_idx in range(total_physical_sheets):
                    # 1. Generate Front Sheet
                    sheet_front = doc_out.new_page(width=sheet_w_pt, height=sheet_h_pt)
                    for r in range(rows):
                        for c in range(cols):
                            slot_idx = int(r * cols + c)
                            p_front = int((sheet_idx * 2) + (slot_idx * pages_per_stack))
                            
                            front_x = float(margin_x + (c * art_w_pt))
                            y = float(margin_y + (r * art_h_pt))
                            r_front = fitz.Rect(front_x, y, front_x + art_w_pt, y + art_h_pt)
                            
                            if p_front < total_input_pages:
                                sheet_front.show_pdf_page(r_front, doc_in, p_front)
                    draw_crop_marks(sheet_front)
                    
                    # 2. Generate Back Sheet (Mirrored)
                    sheet_back = doc_out.new_page(width=sheet_w_pt, height=sheet_h_pt)
                    for r in range(rows):
                        for c in range(cols):
                            slot_idx = int(r * cols + c)
                            p_front = int((sheet_idx * 2) + (slot_idx * pages_per_stack))
                            p_back = int(p_front + 1)
                            
                            back_x = float(margin_x + ((cols - 1 - c) * art_w_pt))
                            y = float(margin_y + (r * art_h_pt))
                            r_back = fitz.Rect(back_x, y, back_x + art_w_pt, y + art_h_pt)
                            
                            if p_back < total_input_pages:
                                sheet_back.show_pdf_page(r_back, doc_in, p_back)
                    draw_crop_marks(sheet_back)
            else:
                total_sheets = int(math.ceil(total_input_pages / float(n_up)))
                for sheet_idx in range(total_sheets):
                    sheet = doc_out.new_page(width=sheet_w_pt, height=sheet_h_pt)
                    for r in range(rows):
                        for c in range(cols):
                            slot_idx = int(r * cols + c)
                            p_target = int(sheet_idx + (slot_idx * total_sheets))
                            
                            x0 = float(margin_x + (c * art_w_pt))
                            y0 = float(margin_y + (r * art_h_pt))
                            r_slot = fitz.Rect(x0, y0, x0 + art_w_pt, y0 + art_h_pt)
                            
                            if p_target < total_input_pages:
                                sheet.show_pdf_page(r_slot, doc_in, p_target)
                    draw_crop_marks(sheet)

        doc_out.save(output_file)
        doc_in.close()
        doc_out.close()
        
        messagebox.showinfo("Success", f"Job complete!\nLayout: {layout_style} ({n_up}-up {side_mode})\nOutput saved.")

    except Exception as e:
        messagebox.showerror("Processing Error", f"A fatal error occurred:\n{str(e)}")

# --- 2. Interface Helper Functions (ADDED) ---
def browse_input():
    filepath = filedialog.askopenfilename(title="Select Input PDF", filetypes=(("PDF files", "*.pdf"), ("All files", "*.*")))
    if filepath:
        input_entry.delete(0, tk.END)
        input_entry.insert(0, filepath)

def browse_output():
    filepath = filedialog.asksaveasfilename(title="Save Output PDF", defaultextension=".pdf", filetypes=(("PDF files", "*.pdf"), ("All files", "*.*")))
    if filepath:
        output_entry.delete(0, tk.END)
        output_entry.insert(0, filepath)

def submit_job():
    # Gather file paths
    input_file = input_entry.get()
    output_file = output_entry.get()
    
    if not input_file or not output_file:
        messagebox.showwarning("Missing Files", "Please specify both an input and an output file path.")
        return

    # Gather and validate dimensions
    try:
        art_w = float(art_w_entry.get())
        art_h = float(art_h_entry.get())
        sheet_w = float(sheet_w_entry.get())
        sheet_h = float(sheet_h_entry.get())
    except ValueError:
        messagebox.showerror("Invalid Input", "Please ensure all dimensions are numerical values.")
        return

    # Gather radio button states
    is_duplex = (duplex_var.get() == "Duplex")
    is_variable = (variable_var.get() == "Variable")
    layout_style = layout_var.get()

    # Pass everything to the engine
    process_imposition(input_file, output_file, art_w, art_h, sheet_w, sheet_h, is_duplex, is_variable, layout_style)


# --- 3. Building the Window Interface ---
root = tk.Tk()
root.title("Universal Imposition Tool")
root.geometry("600x400") 
try:
    root.eval('tk::PlaceWindow . center') 
except tk.TclError:
    pass # Fails silently on some OS window managers if centering isn't supported

duplex_var = tk.StringVar(value="Simplex")
variable_var = tk.StringVar(value="Static")
layout_var = tk.StringVar(value="Cut & Stack")

# File Selection Section
tk.Label(root, text="Input PDF:").grid(row=0, column=0, padx=10, pady=10, sticky="e")
input_entry = tk.Entry(root, width=45)
input_entry.grid(row=0, column=1, columnspan=2)
tk.Button(root, text="Browse", command=browse_input).grid(row=0, column=3, padx=5)

tk.Label(root, text="Output PDF:").grid(row=1, column=0, padx=10, pady=5, sticky="e")
output_entry = tk.Entry(root, width=45)
output_entry.grid(row=1, column=1, columnspan=2)
tk.Button(root, text="Browse", command=browse_output).grid(row=1, column=3, padx=5)

# Dimensions Section
tk.Label(root, text="Artwork (W x H):").grid(row=2, column=0, padx=10, pady=15, sticky="e")
art_w_entry = tk.Entry(root, width=10)
art_w_entry.grid(row=2, column=1, sticky="w")
art_w_entry.insert(0, "8.5") 
tk.Label(root, text="x").grid(row=2, column=1, sticky="e")
art_h_entry = tk.Entry(root, width=10)
art_h_entry.grid(row=2, column=2, sticky="w")
art_h_entry.insert(0, "11.0") 

tk.Label(root, text="Press Sheet (W x H):").grid(row=3, column=0, padx=10, pady=5, sticky="e")
sheet_w_entry = tk.Entry(root, width=10)
sheet_w_entry.grid(row=3, column=1, sticky="w")
sheet_w_entry.insert(0, "12.0") 
tk.Label(root, text="x").grid(row=3, column=1, sticky="e")
sheet_h_entry = tk.Entry(root, width=10)
sheet_h_entry.grid(row=3, column=2, sticky="w")
sheet_h_entry.insert(0, "18.0") 

# Job Properties Section 
tk.Label(root, text="Sides:").grid(row=4, column=0, padx=10, pady=15, sticky="e")
tk.Radiobutton(root, text="Simplex", variable=duplex_var, value="Simplex").grid(row=4, column=1, sticky="w")
tk.Radiobutton(root, text="Duplex", variable=duplex_var, value="Duplex").grid(row=4, column=2, sticky="w")

tk.Label(root, text="Data Type:").grid(row=5, column=0, padx=10, pady=5, sticky="e")
tk.Radiobutton(root, text="Static", variable=variable_var, value="Static").grid(row=5, column=1, sticky="w")
tk.Radiobutton(root, text="Variable", variable=variable_var, value="Variable").grid(row=5, column=2, sticky="w")

# Layout Type Row
tk.Label(root, text="Layout Style:").grid(row=6, column=0, padx=10, pady=5, sticky="e")
tk.Radiobutton(root, text="Cut & Stack", variable=layout_var, value="Cut & Stack").grid(row=6, column=1, sticky="w")
tk.Radiobutton(root, text="Step & Repeat", variable=layout_var, value="Step & Repeat").grid(row=6, column=2, sticky="w")

# Run Button
tk.Button(root, text="Generate Imposition", command=submit_job, bg="#0052cc", fg="white", font=("Arial", 10, "bold")).grid(row=7, column=1, columnspan=2, pady=20)

root.mainloop()

------------------------------
Executing: Step & Repeat (6-up)
Mode: Simplex
------------------------------
Executing: Step & Repeat (6-up)
Mode: Simplex
